# 1. Librerías y config

In [1]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import requests
import urllib3
import io
import re
import numpy as np
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import calendar 
import warnings
import json
import eikon as ek

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'eikon'

# 2. Monitor presente

In [ ]:
# Asegúrate de haber resuelto el problema SSL (actualizar certifi o configurar proxies)
# O si usas verify=False, recuerda la advertencia de seguridad.
# Si estás usando verify=False, descomenta estas líneas para suprimir las advertencias:
# import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Función para obtener cotizaciones (copia la que ya funciona bien para ti)
# Función para obtener cotizaciones
def obtener_cotizaciones():
    url = "https://dolarapi.com/v1/dolares"
    try:
        # Añadimos un timeout. Por ejemplo, 10 segundos.
        # Sigue usando verify=False por ahora, ya que no se solucionó el problema SSL de raíz.
        response = requests.get(url, verify=False, timeout=10) 
        response.raise_for_status() 
        data = response.json()

        if not data:
            print("La API devolvió una lista vacía de cotizaciones.")
            return pd.DataFrame()

        df = pd.DataFrame(data)
        
        df_filtered = pd.DataFrame() 

        if 'casa' in df.columns:
            df_filtered = df[df['casa'].isin(['oficial', 'ccl', 'mep', 'blue'])]
            return df_filtered[['casa', 'compra', 'venta', 'fechaActualizacion']]
        elif 'nombre' in df.columns:
            name_mapping = {
                'Dólar Oficial': 'oficial',
                'Dólar Bolsa': 'mep',
                'Dólar CCL': 'ccl',
                'Dólar Blue': 'blue',
                'Dólar Mayorista': 'mayorista'
            }
            df['casa'] = df['nombre'].map(name_mapping)
            df_filtered = df.dropna(subset=['casa'])
            df_filtered = df_filtered[df_filtered['casa'].isin(['oficial', 'ccl', 'mep', 'blue'])]
            return df_filtered[['casa', 'compra', 'venta', 'fechaActualizacion']]
        else:
            print(f"Error: Ninguna de las columnas esperadas ('casa' o 'nombre') se encontró en la respuesta de la API.")
            print("Columnas disponibles:", df.columns.tolist())
            return pd.DataFrame()

    except requests.exceptions.Timeout:
        print(f"Error de Timeout: La solicitud a DolarAPI tardó demasiado en responder.")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as e:
        print(f"Error de Conexión: No se pudo establecer la conexión a DolarAPI. Posible problema de red o DNS. {e}")
        return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Error al obtener datos de DolarAPI: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Ocurrió un error inesperado al procesar los datos: {e}")
        return pd.DataFrame()

df_dolares = obtener_cotizaciones()

if not df_dolares.empty:
    print("Datos disponibles:")
    print(df_dolares.to_string(index=False))
    print("-" * 30)

    oficial_venta = None
    ccl_venta = None

    if 'oficial' in df_dolares['casa'].values:
        oficial_venta = df_dolares[df_dolares['casa'] == 'oficial']['venta'].values[0]
        print(f"Dólar Oficial: ${oficial_venta:.2f}")
    else:
        print("Advertencia: La cotización del Dólar Oficial no está disponible.")

    if 'ccl' in df_dolares['casa'].values:
        ccl_venta = df_dolares[df_dolares['casa'] == 'ccl']['venta'].values[0]
        print(f"Dólar CCL: ${ccl_venta:.2f}")
    else:
        print("Advertencia: La cotización del Dólar CCL no está disponible. No se puede calcular la brecha CCL.")

    if oficial_venta is not None and ccl_venta is not None:
        brecha_ccl = ((ccl_venta / oficial_venta) - 1) * 100
        print(f"Brecha CCL: {brecha_ccl:.2f}%")
    else:
        print("No se puede calcular la Brecha CCL debido a la falta de cotizaciones.")
        print("Considera usar el Dólar MEP si el CCL no está disponible o buscar una fuente alternativa para CCL.")

else:
    print("No se pudieron obtener datos de cotización o el DataFrame resultante está vacío. Revisa los mensajes de error anteriores.")


# ITCRM

In [ ]:
# Desactivar advertencias SSL del BCRA
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def obtener_itcrm_actualizado():
    print("1. Entrando a la web del ITCRM del BCRA...")
    
    # URL de la página donde están los datos
    url_web = "https://www.bcra.gob.ar/PublicacionesEstadisticas/Indices_tipo_cambio_multilateral.asp"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }
    
    try:
        # Paso 1: Buscar la URL del Excel en la página
        res_web = requests.get(url_web, headers=headers, verify=False, timeout=15)
        res_web.raise_for_status()
        
        # Buscamos cualquier link que sea ITCRMSerie.xls o .xlsx
        links = re.findall(r'href="([^"]+ITCRMSerie\.xlsx?)"', res_web.text, re.IGNORECASE)
        
        if links:
            ruta_parcial = links[0]
            url_excel = "https://www.bcra.gob.ar" + ruta_parcial if ruta_parcial.startswith("/") else ruta_parcial
            print(f"✅ ¡Excel del ITCRM encontrado!: {url_excel}")
        else:
            # Fallback de seguridad
            url_excel = "https://www.bcra.gob.ar/archivos/Pdfs/PublicacionesEstadisticas/ITCRMSerie.xlsx"
            print(f"⚠️ Link no encontrado en HTML, usando enlace directo: {url_excel}")

        # Paso 2: Descargar el Excel
        print("2. Descargando el archivo Excel a la memoria...")
        res_excel = requests.get(url_excel, headers=headers, verify=False, timeout=15)
        res_excel.raise_for_status()
        
        # Paso 3: Leer los datos con Pandas
        print("3. Procesando los datos del Tipo de Cambio Real Multilateral...")
        
        # --- LA CORRECCIÓN ESTÁ AQUÍ ---
        # Dejamos que Pandas decida el motor automáticamente (openpyxl para xlsx)
        try:
            df_raw = pd.read_excel(io.BytesIO(res_excel.content), header=None)
        except ValueError:
            # Si llega a quejarse porque en realidad era un .xls viejo, usamos el motor viejo
            df_raw = pd.read_excel(io.BytesIO(res_excel.content), engine='xlrd', header=None)
        # -------------------------------

        # El archivo del ITCRM tiene texto inútil arriba. Buscamos dónde empieza la tabla.
        columna_textos = df_raw[0].astype(str).str.lower()
        idx_inicio = columna_textos[columna_textos.str.contains("período|fecha|itcrm")].index
        
        if len(idx_inicio) > 0:
            fila_nombres = idx_inicio[0]
            # Cortamos el dataframe desde donde empiezan los datos reales
            df_itcrm = df_raw.iloc[fila_nombres + 1:].copy()
        else:
            df_itcrm = df_raw.iloc[2:].copy()
            
        # Nos quedamos SOLO con las dos primeras columnas (Fecha y Valor ITCRM)
        df_itcrm = df_itcrm.iloc[:, [0, 1]]
        df_itcrm.columns = ['Fecha', 'ITCRM_Valor']
        
        # Limpieza final: Eliminar filas vacías o textos basura al final del Excel
        df_itcrm = df_itcrm.dropna(subset=['Fecha', 'ITCRM_Valor'])
        df_itcrm = df_itcrm[pd.to_numeric(df_itcrm['ITCRM_Valor'], errors='coerce').notnull()]
        
        # Convertir a Fecha de Python y fijarla como índice
        df_itcrm['Fecha'] = pd.to_datetime(df_itcrm['Fecha'], errors='coerce')
        df_itcrm = df_itcrm.dropna(subset=['Fecha']) 
        df_itcrm['ITCRM_Valor'] = df_itcrm['ITCRM_Valor'].astype(float)
        
        df_itcrm = df_itcrm.set_index('Fecha').sort_index()
        
        print(f"✅ ¡Éxito! Se descargó la serie desde {df_itcrm.index.min().strftime('%d/%m/%Y')} hasta {df_itcrm.index.max().strftime('%d/%m/%Y')}")
        return df_itcrm

    except Exception as e:
        print(f"❌ Error al intentar extraer los datos del ITCRM: {e}")
        return pd.DataFrame()

# ==========================================
# EJECUCIÓN
# ==========================================
df_mi_itcrm = obtener_itcrm_actualizado()

display(df_mi_itcrm.tail())

In [ ]:
# ==================================================
# 📌 TERMÓMETRO DEL PRODUCTOR (Costo de Oportunidad)
# ==================================================

# 1. Obtenemos las métricas clave de la serie histórica
ultimo_valor_itcrm = df_mi_itcrm['ITCRM_Valor'].iloc[-1]
fecha_ultimo = df_mi_itcrm.index[-1].strftime('%d/%m/%Y')

promedio_historico = df_mi_itcrm['ITCRM_Valor'].mean()
cuartil_bajo = df_mi_itcrm['ITCRM_Valor'].quantile(0.25)
cuartil_alto = df_mi_itcrm['ITCRM_Valor'].quantile(0.75)

print(f"📅 Fecha de análisis: {fecha_ultimo}")
print(f"📊 Valor Actual ITCRM: {ultimo_valor_itcrm:.1f}")
print(f"📈 Promedio Histórico: {promedio_historico:.1f}\n")

# 2. Lógica del Semáforo Comercial
if ultimo_valor_itcrm < cuartil_bajo:
    print("🚨 ZONA ROJA (Atraso Cambiario Fuerte)")
    print("El tipo de cambio real está muy por debajo de la media histórica.")
    print("💡 COMPORTAMIENTO ESPERADO: El productor tratará de NO fijar precio (PAF abierto) y retener grano físico como cobertura, a menos que consiga tasas de interés en pesos muy altas (Carry Trade).")

elif ultimo_valor_itcrm > cuartil_alto:
    print("🟢 ZONA VERDE (Tipo de Cambio Competitivo)")
    print("El tipo de cambio real es altísimo en términos históricos.")
    print("💡 COMPORTAMIENTO ESPERADO: Fuerte aceleración en la fijación de precios (Pricing) y venta de físico. El productor aprovecha el poder adquisitivo actual.")

else:
    print("⚖️ ZONA NEUTRAL (Promedios Históricos)")
    print("El tipo de cambio no es un driver ni a favor ni en contra.")
    print("💡 COMPORTAMIENTO ESPERADO: La decisión de fijar precios dependerá estrictamente de sus necesidades de caja para pagar insumos, deudas o la próxima siembra.")

print("==================================================")

In [ ]:
# 1. Preparar las métricas de referencia
ultimo_valor = df_mi_itcrm['ITCRM_Valor'].iloc[-1]
fecha_ultima = df_mi_itcrm.index[-1].strftime('%d-%b-%Y')
promedio = df_mi_itcrm['ITCRM_Valor'].mean()
p25 = df_mi_itcrm['ITCRM_Valor'].quantile(0.25) # Cuartil bajo (zona de atraso)
p75 = df_mi_itcrm['ITCRM_Valor'].quantile(0.75) # Cuartil alto (zona de competitividad)

# 2. Configuración general del estilo
sns.set_theme(style="whitegrid", rc={"axes.edgecolor": "0.15", "axes.linewidth": 1.25})
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'🌡️ Dashboard de Tipo de Cambio Real Multilateral (ITCRM) - Actualizado al {fecha_ultima}', 
             fontsize=20, fontweight='bold', y=0.98)

# Crear una grilla (1 gráfico arriba ocupando todo el ancho, 2 abajo)
gs = gridspec.GridSpec(2, 2, height_ratios=[1.2, 1], hspace=0.3, wspace=0.2)

# ==========================================
# GRÁFICO 1: SERIE HISTÓRICA COMPLETA (Arriba)
# ==========================================
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df_mi_itcrm.index, df_mi_itcrm['ITCRM_Valor'], color='#1f77b4', linewidth=1.5)

# Líneas de referencia
ax1.axhline(promedio, color='gray', linestyle='--', linewidth=2, label=f'Promedio Histórico ({promedio:.1f})')
ax1.axhline(p25, color='crimson', linestyle=':', linewidth=2, label=f'Cuartil Bajo - Atraso ({p25:.1f})')
ax1.axhline(p75, color='forestgreen', linestyle=':', linewidth=2, label=f'Cuartil Alto - Competitivo ({p75:.1f})')

# Sombrear la zona "Neutral" (entre el percentil 25 y 75)
ax1.fill_between(df_mi_itcrm.index, p25, p75, color='gray', alpha=0.1, label='Zona Neutral')

# Marcar el punto actual
ax1.plot(df_mi_itcrm.index[-1], ultimo_valor, marker='o', markersize=10, color='red')
ax1.annotate(f'HOY: {ultimo_valor:.1f}', 
             xy=(df_mi_itcrm.index[-1], ultimo_valor), 
             xytext=(df_mi_itcrm.index[-1] + pd.Timedelta(days=600), ultimo_valor - 20), # Ajuste manual para que no se superponga
             textcoords='data', # Usar coordenadas de datos para xytext
             fontsize=12, fontweight='bold', color='red',
             arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax1.set_title("Evolución Histórica Completa (1997 - Presente)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Valor ITCRM (Base 100)")
ax1.legend(loc='upper left', fontsize=10) # Cambié la ubicación para que no tape el gráfico

# ==========================================
# GRÁFICO 2: ZOOM RECIENTE (Abajo Izquierda)
# ==========================================
ax2 = fig.add_subplot(gs[1, 0])
# Filtramos desde enero 2020
df_reciente = df_mi_itcrm[df_mi_itcrm.index >= '2020-01-01']

ax2.plot(df_reciente.index, df_reciente['ITCRM_Valor'], color='darkorange', linewidth=2)
ax2.axhline(ultimo_valor, color='red', linestyle='--', alpha=0.7, label=f'Valor Actual ({ultimo_valor:.1f})')

# Formato de fechas para el eje X
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

ax2.set_title("Zoom: Dinámica Reciente (Desde 2020)", fontsize=14, fontweight='bold')
ax2.set_ylabel("Valor ITCRM")
ax2.legend(loc='lower left', fontsize=10) # Añadir leyenda del valor actual

# ==========================================
# GRÁFICO 3: DISTRIBUCIÓN HISTÓRICA MEJORADA (Abajo Derecha)
# ==========================================
ax3 = fig.add_subplot(gs[1, 1])

# 1. Calculamos los bins una única vez para TODO el rango de datos
# Esto asegura que todas las llamadas a histplot usen las mismas divisiones
bins = 60 
min_val_total = df_mi_itcrm['ITCRM_Valor'].min()
max_val_total = df_mi_itcrm['ITCRM_Valor'].max()
# np.linspace crea divisiones uniformes desde el mínimo hasta el máximo
common_bins = np.linspace(min_val_total, max_val_total, bins + 1) 

# 2. Graficar las distintas zonas de la distribución con colores,
#    pasando los 'common_bins' en lugar de 'range'
sns.histplot(df_mi_itcrm['ITCRM_Valor'][df_mi_itcrm['ITCRM_Valor'] < p25], 
             bins=common_bins, # Usamos los bins calculados
             color='crimson', alpha=0.5, label='Atraso (< Q25)', 
             ax=ax3, stat='count', edgecolor='white')

sns.histplot(df_mi_itcrm['ITCRM_Valor'][(df_mi_itcrm['ITCRM_Valor'] >= p25) & (df_mi_itcrm['ITCRM_Valor'] < p75)], 
             bins=common_bins, # Usamos los bins calculados
             color='gray', alpha=0.3, label='Neutral (Q25-Q75)', 
             ax=ax3, stat='count', edgecolor='white')

sns.histplot(df_mi_itcrm['ITCRM_Valor'][df_mi_itcrm['ITCRM_Valor'] >= p75], 
             bins=common_bins, # Usamos los bins calculados
             color='forestgreen', alpha=0.5, label='Competitivo (> Q75)', 
             ax=ax3, stat='count', edgecolor='white')

# Añadir la estimación de densidad (KDE) sobre todos los datos
sns.kdeplot(df_mi_itcrm['ITCRM_Valor'], color='blue', ax=ax3, linewidth=2, label='Densidad Estimada')

# Línea marcando el promedio histórico
ax3.axvline(promedio, color='gray', linestyle='--', linewidth=2, label=f'Promedio Histórico ({promedio:.1f})')

# Línea y anotación del valor actual
ax3.axvline(ultimo_valor, color='red', linestyle='-', linewidth=3, label=f'Valor Actual ({ultimo_valor:.1f})')

# Anotación mejorada y reubicada para evitar superposición
# Ajustamos la posición xytext para que el texto no se superponga con la flecha y se vea más claro
ax3.annotate(f'Hoy: {ultimo_valor:.1f}', 
             xy=(ultimo_valor, ax3.get_ylim()[1]*0.8), # Punto de origen de la flecha
             xytext=(ultimo_valor + 0.1 * (max_val_total - min_val_total), ax3.get_ylim()[1]*0.9), # Posición del texto
             fontsize=12, fontweight='bold', color='red',
             arrowprops=dict(arrowstyle='->', color='red', lw=2, connectionstyle='arc3,rad=.2')) 


ax3.set_title("Distribución Histórica: ¿Qué tan raro es el valor de hoy?", fontsize=14, fontweight='bold')
ax3.set_xlabel("Valor ITCRM (Base 100)")
ax3.set_ylabel("Frecuencia (Días)")
ax3.legend(loc='upper right', fontsize=10) # Mostrar la leyenda de las zonas

# Ajustes finales
sns.despine() # Elimina los bordes superiores y derechos
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajusta el layout para que el título no se superponga
plt.show()

# Curva futuros rofex - CP

In [ ]:
# 1. Conexión local a tu Refinitiv Workspace (DEBE ESTAR ABIERTO)
# Reemplaza el texto entre comillas con tu clave real
ek.set_app_key('7b98b2892832466797470f23246a20cfd5c0f9b5')

print("Conectado localmente. Descargando la cadena de futuros de Soja...")
# Intenta conectarte especificando el puerto
ek.set_port_number(9000)
# 2. Descargar datos
df, err = ek.get_data("0#S:", ["DSPLY_NAME", "EXPIR_DATE", "CF_LAST", "SETTLE"])

if not df.empty:
    # 3. Preparación de los datos
    df['Precio'] = df['CF_LAST'].fillna(df['SETTLE'])
    df_clean = df.dropna(subset=['EXPIR_DATE', 'Precio']).copy()
    
    # Arreglamos el formato numérico y de fecha
    df_clean['Precio'] = df_clean['Precio'].astype(float)
    df_clean['EXPIR_DATE'] = pd.to_datetime(df_clean['EXPIR_DATE'], errors='coerce')
    df_clean = df_clean.dropna(subset=['EXPIR_DATE'])
    df_clean = df_clean.sort_values('EXPIR_DATE')
    
    # 4. Graficar la curva
    plt.figure(figsize=(12, 6))
    
    plt.plot(df_clean['EXPIR_DATE'], df_clean['Precio'], 
             marker='o', linestyle='-', color='#2ca02c', linewidth=2.5)
    
    plt.title('Curva de Futuros de Soja Local (CBOT)', fontsize=16, fontweight='bold')
    plt.xlabel('Fecha de Vencimiento', fontsize=12)
    plt.ylabel('Precio (USD / Bushel)', fontsize=12)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    
    # Etiquetas
    for i, row in df_clean.iterrows():
        plt.annotate(row['Instrument'], (row['EXPIR_DATE'], row['Precio']), 
                     textcoords="offset points", xytext=(0,12), ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print("No se descargaron datos.")

2026-03-12 17:04:27,503 P[24496] [MainThread 26256] Error: no proxy address identified.
Check if Eikon Desktop or Eikon API Proxy is running.
2026-03-12 17:04:27,505 P[24496] [MainThread 26256] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2026-03-12 17:04:27,505 P[24496] [MainThread 26256] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2026-03-12 17:04:27,506 P[24496] [MainThread 26256] Port number was not identified, cannot send any request
2026-03-12 17:04:27,508 P[24496] [MainThread 26256] HTTP request failed: Invalid port: 'None'


Conectado localmente. Descargando la cadena de futuros de Soja...


AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
import eikon as ek

# Pon tu clave aquí
ek.set_app_key('7b98b2892832466797470f23246a20cfd5c0f9b5')

try:
    puerto = ek.get_port_number()
    print(f"✅ ¡Conexión exitosa! Workspace está escuchando en el puerto: {puerto}")
except Exception as e:
    print(f"❌ Falló la conexión. Error: {e}")

2026-03-12 17:04:39,295 P[24496] [MainThread 26256] Error: no proxy address identified.
Check if Eikon Desktop or Eikon API Proxy is running.
2026-03-12 17:04:39,296 P[24496] [MainThread 26256] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2026-03-12 17:04:39,298 P[24496] [MainThread 26256] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2026-03-12 17:04:39,299 P[24496] [MainThread 26256] Port number was not identified, cannot send any request


✅ ¡Conexión exitosa! Workspace está escuchando en el puerto: None


# REM - LP

In [ ]:
# Desactivar advertencias de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def obtener_rem_completo():
    print("1. Entrando a la página del BCRA como 'Scraper'...")
    url_home = "https://www.bcra.gob.ar/PublicacionesEstadisticas/Relevamiento_Expectativas_de_Mercado.asp"
    
    try:
        res_home = requests.get(url_home, verify=False, timeout=15)
        res_home.raise_for_status()
        
        links = re.findall(r'href="([^"]+\.xlsx)"', res_home.text, re.IGNORECASE)
        url_excel = None
        for link in links:
            if "hist" in link.lower() or "result" in link.lower():
                url_excel = "https://www.bcra.gob.ar" + link if link.startswith("/") else link
                break
                
        if not url_excel:
            print("❌ No se encontró el Excel en la web.")
            return pd.DataFrame()
            
        print(f"✅ ¡URL capturada!: {url_excel}")
        
    except Exception as e:
        print(f"❌ Error en la web: {e}")
        return pd.DataFrame()

    print("2. Descargando el archivo Excel a la memoria...")
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(url_excel, headers=headers, verify=False, timeout=15)
        response.raise_for_status() 
        excel_data = io.BytesIO(response.content)
        
        print("3. Extrayendo TODA la macroeconomía del REM...")
        
        df_raw = pd.read_excel(excel_data, header=None)
        fechas = df_raw.iloc[1, 1:].values
        nombres_variables = df_raw[0].astype(str).str.strip()
        
        def limpiar_numero_excel(x):
            if pd.isna(x) or str(x).strip() == '' or str(x).strip().lower() == 'nan': return np.nan
            if isinstance(x, (int, float)): return float(x)
            try: return float(str(x).replace('.', '').replace(',', '.'))
            except: return np.nan

        # =========================================================
        # EL MOTOR DE EXTRACCIÓN: Diccionario de variables y regex
        # =========================================================
        diccionario_busqueda = {
            'Proy_Inflacion_12M': r"IPC nivel general.*Próx\. 12 meses",
            'Proy_Inflacion_24M': r"IPC nivel general.*Próx\. 24 meses",
            'Proy_Inflacion_Nucleo_12M': r"IPC núcleo.*Próx\. 12 meses",
            'Proy_Dolar_12M': r"Tipo de cambio nominal.*Próx\. 12 meses",
            'Proy_Tasa_Interes_12M': r"Tasa de interés.*Próx\. 12 meses"
        }
        
        # Diccionario base para armar el nuevo DataFrame
        datos_extraidos = {'Mes_Relevamiento': fechas}
        
        # Iteramos dinámicamente por cada variable que queremos
        for nombre_columna, patron_regex in diccionario_busqueda.items():
            match = nombres_variables[nombres_variables.str.contains(patron_regex, regex=True, na=False)]
            if not match.empty:
                idx = match.index[0]
                fila_mediana = df_raw.iloc[idx + 1, 1:].values # +1 es la fila de la "Mediana"
                datos_extraidos[nombre_columna] = [limpiar_numero_excel(x) for x in fila_mediana]
            else:
                print(f"⚠️ Advertencia: No se encontró la fila para {nombre_columna}")
                datos_extraidos[nombre_columna] = [np.nan] * len(fechas)

        # Armamos el DataFrame
        df_rem = pd.DataFrame(datos_extraidos)
        df_rem = df_rem.dropna(subset=['Mes_Relevamiento'])
        
        # Convertidor de Fechas robusto
        def parsear_fecha(f):
            if pd.isna(f) or str(f).lower() == 'nan': return pd.NaT
            if isinstance(f, pd.Timestamp) or hasattr(f, 'to_pydatetime'): return pd.to_datetime(f)
            f_str = str(f).strip().lower()
            if f_str.startswith('20') and len(f_str) >= 10: return pd.to_datetime(f_str[:10], errors='coerce')
            meses_es_to_en = {'ene': 'Jan', 'feb': 'Feb', 'mar': 'Mar', 'abr': 'Apr', 'may': 'May', 'jun': 'Jun', 'jul': 'Jul', 'ago': 'Aug', 'sep': 'Sep', 'oct': 'Oct', 'nov': 'Nov', 'dic': 'Dec'}
            mes = f_str[:3]
            anio = f_str[-2:]
            mes_en = meses_es_to_en.get(mes)
            if mes_en: return pd.to_datetime(f"{mes_en}-{anio}", format="%b-%y", errors='coerce')
            return pd.to_datetime(f_str, errors='coerce')

        df_rem['Fecha'] = df_rem['Mes_Relevamiento'].apply(parsear_fecha)
        df_rem = df_rem.dropna(subset=['Fecha']).drop(columns=['Mes_Relevamiento']).set_index('Fecha')
        
        # =========================================================
        # CÁLCULOS FINANCIEROS DE VALOR AGREGADO
        # =========================================================
        # Tasa Real (Ecuación de Fisher: ((1 + i) / (1 + pi)) - 1)
        if 'Proy_Tasa_Interes_12M' in df_rem.columns and 'Proy_Inflacion_12M' in df_rem.columns:
            tasa = df_rem['Proy_Tasa_Interes_12M'] / 100
            infla = df_rem['Proy_Inflacion_12M'] / 100
            df_rem['Proy_Tasa_Real_12M'] = ((1 + tasa) / (1 + infla) - 1) * 100
        
        print("✅ ¡Éxito total! Macroeconomía completa procesada.")
        return df_rem

    except Exception as e:
        print(f"❌ Error al extraer los datos del REM: {e}")
        return pd.DataFrame()

# Ejecución y muestra
df_mi_rem = obtener_rem_completo()
display(df_mi_rem.tail())